In [ ]:
#Load dataset
from google.colab import drive
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Provide the path to your CSV file in Google Drive
file_path = '/content/drive/MyDrive/cumulative.csv'  # Update path as needed
df = pd.read_csv(file_path)

# Display the first 5 rows
df.head()

print(f"Initial Shape: {df.shape}")
print("Target Distribution:\n", df['koi_disposition'].value_counts())

Mounted at /content/drive
Initial Shape: (9564, 50)
Target Distribution:
 koi_disposition
FALSE POSITIVE    5023
CONFIRMED         2293
CANDIDATE         2248
Name: count, dtype: int64


In [ ]:
# Data Cleaning & Feature Filtering
import pandas as pd
import numpy as np

def preprocess_kepler_data(df_input):
    df = df_input.copy()

    # 1. Filter target classes (CONFIRMED=1, FALSE POSITIVE=0)
    df = df[df['koi_disposition'].isin(['CONFIRMED', 'FALSE POSITIVE'])].copy()
    df['target'] = (df['koi_disposition'] == 'CONFIRMED').astype(int)

    # 2. Drop direct target leakage columns (labels and disposition scores)
    drop_leakage = [
        'kepid', 'kepoi_name', 'kepler_name', 'koi_disposition',
        'koi_pdisposition', 'rowid', 'koi_score',
        'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec'
    ]
    df = df.drop(columns=[c for c in drop_leakage if c in df.columns])

    # 3. Drop columns with >40% missing values
    missing_ratio = df.isnull().sum() / len(df)
    df = df.drop(columns=missing_ratio[missing_ratio > 0.4].index)

    # 4. Select strictly numeric features
    X = df.drop(columns=['target']).select_dtypes(include=[np.number])
    y = df['target']

    return X, y

# Run Step 2 (raw split ready)
X, y = preprocess_kepler_data(df)
print(f"Raw Dataset Filtered: {X.shape[0]} records x {X.shape[1]} numeric features")

Raw Dataset Filtered: 7316 records x 36 numeric features


In [ ]:
# Train/Test Split, Clean Imputation & 100k Upsampling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import pandas as pd

# 1. Split FIRST (80% Train, 20% Test) on original real rows
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Calculate missing value medians ONLY on training set (Prevents Imputation Leakage)
train_medians = X_train_raw.median(numeric_only=True)
X_train_clean = X_train_raw.fillna(train_medians)
X_test_clean = X_test_raw.fillna(train_medians)

# 3. Create 100k upsampled training dataset for final model fitting
train_combined = pd.concat([X_train_clean, y_train_raw], axis=1)
train_100k = resample(train_combined, replace=True, n_samples=100000, random_state=42)

X_train_100k = train_100k.drop(columns=[y_train_raw.name])
y_train_100k = train_100k[y_train_raw.name]

# 4. Fit Scaler ONLY on training data
scaler = StandardScaler()
# Scaled datasets for Cross-Validation (Real ~5.8k rows)
X_train_cv_scaled = pd.DataFrame(scaler.fit_transform(X_train_clean), columns=X_train_clean.columns)

# Scaled datasets for Final 100k Training and Clean Testing
X_train_100k_scaled = pd.DataFrame(scaler.transform(X_train_100k), columns=X_train_100k.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_clean), columns=X_test_clean.columns)

print(f"Clean Real Train CV Shape: {X_train_cv_scaled.shape}")
print(f"Final Train Shape (Upsampled): {X_train_100k_scaled.shape}")
print(f"Test Shape (Clean Holdout):   {X_test_scaled.shape}")

Clean Real Train CV Shape: (5852, 36)
Final Train Shape (Upsampled): (100000, 36)
Test Shape (Clean Holdout):   (1464, 36)


In [ ]:
# Model Training & Hyperparameter Fine-Tuning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV

print("Training & Fine-Tuning 4 Models...")

# 1. Logistic Regression
print("\n[1/4] Training Logistic Regression...")
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_model.fit(X_train_100k_scaled, y_train_100k)

# 2. Random Forest (Search on CV data, Fit on 100k)
print("[2/4] Fine-Tuning Random Forest...")
rf_param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [8, 12, 16],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4]
}
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_search = RandomizedSearchCV(
    rf_base, param_distributions=rf_param_grid, n_iter=4,
    cv=3, scoring='f1', random_state=42, n_jobs=-1
)
rf_search.fit(X_train_cv_scaled, y_train_raw)
best_rf = rf_search.best_estimator_
best_rf.fit(X_train_100k_scaled, y_train_100k)

# 3. XGBoost (Search on CV data, Fit on 100k)
print("[3/4] Fine-Tuning XGBoost...")
xgb_param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.08],
    'subsample': [0.7, 0.85]
}
xgb_base = XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)
xgb_search = RandomizedSearchCV(
    xgb_base, param_distributions=xgb_param_grid, n_iter=4,
    cv=3, scoring='f1', random_state=42, n_jobs=-1
)
xgb_search.fit(X_train_cv_scaled, y_train_raw)
best_xgb = xgb_search.best_estimator_
best_xgb.fit(X_train_100k_scaled, y_train_100k)

# 4. LightGBM (Search on CV data, Fit on 100k)
print("[4/4] Fine-Tuning LightGBM...")
lgb_param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.08],
    'num_leaves': [15, 31]
}
lgb_base = LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1)
lgb_search = RandomizedSearchCV(
    lgb_base, param_distributions=lgb_param_grid, n_iter=4,
    cv=3, scoring='f1', random_state=42, n_jobs=-1
)
lgb_search.fit(X_train_cv_scaled, y_train_raw)
best_lgb = lgb_search.best_estimator_
best_lgb.fit(X_train_100k_scaled, y_train_100k)

print("\nAll 4 models fine-tuned without CV leakage & trained on 100k rows!")

Training & Fine-Tuning 4 Models...

[1/4] Training Logistic Regression...
[2/4] Fine-Tuning Random Forest...
[3/4] Fine-Tuning XGBoost...
[4/4] Fine-Tuning LightGBM...

All 4 models fine-tuned without CV leakage & trained on 100k rows!


In [ ]:
# Metric Evaluation & Performance Comparison
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

models = {
    "Logistic Regression": lr_model,
    "Random Forest": best_rf,
    "XGBoost": best_xgb,
    "LightGBM": best_lgb
}

results = {}

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test_raw, y_pred),
        "Precision": precision_score(y_test_raw, y_pred),
        "Recall": recall_score(y_test_raw, y_pred),
        "F1-Score": f1_score(y_test_raw, y_pred),
        "ROC-AUC": roc_auc_score(y_test_raw, y_prob)
    }

comparison_df = pd.DataFrame(results).T
print("                  - REFINED PIPELINE COMPARISON -")
print("")
print(comparison_df.map(lambda x: f"{x * 100:.2f}%"))

                  - REFINED PIPELINE COMPARISON -

                    Accuracy Precision  Recall F1-Score ROC-AUC
Logistic Regression   91.05%    87.10%  83.88%   85.46%  96.70%
Random Forest         93.92%    91.48%  88.89%   90.17%  97.97%
XGBoost               94.67%    93.20%  89.54%   91.33%  98.59%
LightGBM              94.67%    93.39%  89.32%   91.31%  98.57%


In [ ]:
# Metric Evaluation & Performance Comparison
import pickle
from sklearn.pipeline import Pipeline
from google.colab import files

best_model_name = max(results, key=lambda k: results[k]['F1-Score'])
winning_model = models[best_model_name]

deployment_pipeline = Pipeline(steps=[
    ('scaler', scaler),
    ('classifier', winning_model)
])

pickle_filename = "kepler_exoplanet_model.pkl"
with open(pickle_filename, 'wb') as file:
    pickle.dump(deployment_pipeline, file)

print(f"Saved pipeline with ({best_model_name}) to '{pickle_filename}'.")
files.download(pickle_filename)

Saved pipeline with (XGBoost) to 'kepler_exoplanet_model.pkl'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>